# Case Study — Where Humans and LLM4BEAR Diverge

Each human-evaluation survey page showed a **BundleRec (original)** bundle side-by-side with the
**LLM4BEAR (refined)** bundle. Participants scored each `/5` **without ever seeing the LLM's design intent**.

This notebook isolates the sharpest disagreements — pairs where a **human rated the LLM4BEAR bundle 1 or 2 / 5**,
even though:

* **LLM4BEAR** raised its own score for that bundle (its refinement *self-rating*), and
* a **blind Claude re-rating** (given both bundles, no intent) also scored the LLM4BEAR bundle **4 or 5 / 5**.

For each case we surface the **LLM's design intent** (which participants never saw) so the divergence can be
written up qualitatively. We restrict to **3–4 item** LLM4BEAR bundles and show **5 cases per domain**.

> `randomised` in the surveys only means the on-screen left/right positions of the two bundles were swapped;
> it is used here solely to identify which displayed side is the LLM4BEAR bundle.

In [1]:
import pandas as pd, numpy as np, pickle, json, re, os
from bs4 import BeautifulSoup, Comment
from IPython.display import HTML, display

In [ ]:
# Clone the repo (same sparse checkout as the sibling notebooks)
!rm -rf LLM4BEAR
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git
!cd LLM4BEAR && git sparse-checkout set "BundleRec Data"
!cd LLM4BEAR && git sparse-checkout add "3_Human Evaluation"

Cloning into 'LLM4BEAR'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 81 (delta 11), reused 43 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (81/81), 43.95 KiB | 6.28 MiB/s, done.
Resolving deltas: 100% (11/11), done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 9.90 KiB | 3.30 MiB/s, done.
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 82 (delta 7), reused 80 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 46.13 MiB | 8.73 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Updating files: 100% (165/165), done.
remote: Enumerating objects: 450, done.
remote: Counting objects: 100% (450/450), done.
remote: Compressing ob

In [3]:
BASE = "/content/LLM4BEAR"          # change if you cloned elsewhere
HE   = f"{BASE}/3_Human Evaluation"
BR   = f"{BASE}/BundleRec Data"

# Forms actually completed by participants (from the analysis notebook)
clothing_forms   = [1,3,4,5,6,7,10,11,13,15,16,17,18,19,20,21,22,24,25,29,30,31,32,33,34,35,37,38]
electronic_forms = [1,2,3,4,5,6,7,8,9,10,12,13,14,15,16,17,18,20,22,23,24,25,26,27,28,29,30,31,32,34,35,36,39]
food_forms       = [1,2,4,5,6,7,9,10,11,12,14,15,19,20,21,23,24,27,28,31,33,34,35,36,37,38]
FORMS  = {"clothing": clothing_forms, "electronic": electronic_forms, "food": food_forms}

# Naming quirk: forms / metadata / thumbs use the singular ("electronic"),
# while surveys / randomise_mapping / claude pkl use the plural ("electronics").
PLURAL = {"clothing": "clothing", "electronic": "electronics", "food": "food"}

DOMAINS = ["clothing", "electronic", "food"]

# Selection knobs
LOW_SCORES   = (1.0, 2.0)   # human score on the LLM4BEAR bundle counts as "diverged"
CLAUDE_HIGH  = (4, 5)       # require Claude to have rated the LLM4BEAR bundle this high
BUNDLE_SIZES = (3, 4)       # LLM4BEAR bundle size
N_PER_DOMAIN = 5

## 1 · Reconstruct the LLM4BEAR bundles, intents and LLM self-ratings

The refinement pkl stores 21 iterations. Surveyed pairs are the bundles that started flagged and finished
un-flagged **and** whose item set actually changed (`cleaned_mod_indices`) — the exact same construction the
survey generator used, so global pair `P` maps to `cleaned_mod_indices[P]`. Iteration `[0]` is the original
(BundleRec) bundle, iteration `[-1]` is the final LLM4BEAR bundle; `scores` are on a `1–5` scale.

In [4]:
def flag_guys(flags):
    first, last = flags[0], flags[-1]
    return [i for i in range(len(first)) if first[i] and not last[i]]

def load_domain(dom):
    with open(f"{HE}/historic_bundle_refinement/historical_bundle_changes_{dom}_complete_{dom}_no_graph_help_run.pkl","rb") as f:
        intents, bundle_items, bundle_indices, scores, min_scores, flags = pickle.load(f)
    mod     = flag_guys(flags)
    cleaned = [i for i in mod if set(bundle_indices[0][i]) != set(bundle_indices[-1][i])]

    titles = list(pd.read_csv(f"{BR}/{dom}/merged_metadata.csv")["titles"])
    with open(f"{BR}/enriched_outputs_{dom}.pkl","rb") as f:
        descs = pickle.load(f)
    thumbs = pd.read_csv(f"{HE}/{dom}_files_sorted_thumbs.csv")
    def num_from_name(n):
        m = re.search(r"(\d+)", str(n)); return int(m.group(1)) if m else None
    num2url = {num_from_name(r["name"]): r["imageLink"] for _, r in thumbs.iterrows()}

    return dict(intents=intents, bundle_indices=bundle_indices, scores=scores,
                cleaned=cleaned, titles=titles, descs=descs, num2url=num2url)

def idx_to_url(D, i):  return D["num2url"][i + 1]      # filenames are 1-based
def drive_id(url):
    m = re.search(r"[?&]id=([-\w]+)", str(url)); return m.group(1) if m else None

DATA = {dom: load_domain(dom) for dom in DOMAINS}
for dom in DOMAINS:
    print(f"{dom:11s}: {len(DATA[dom]['cleaned'])} surveyed LLM4BEAR bundles reconstructed")

clothing   : 933 surveyed LLM4BEAR bundles reconstructed
electronic : 1236 surveyed LLM4BEAR bundles reconstructed
food       : 1229 surveyed LLM4BEAR bundles reconstructed


## 2 · Human ratings (with identifiers) and the blind Claude ratings

Each completed form holds one participant's 20 pair-ratings. `randomised == "yes"` means **Bundle 1** on that
page was the LLM4BEAR bundle. The Claude re-ratings live in `{domain}_claude_zero_shot_responses.pkl`, ordered by
position in the forms list (`k = f*20 + j`); we decode the LLM4BEAR-side rating with the same swap logic.

In [5]:
def parse_forms(dom):
    # For each completed form -> list of 20 pair dicts with human ratings + identifiers.
    out = []
    for batch_i in FORMS[dom]:
        form = pd.read_csv(f"{HE}/forms/results_{dom}_{batch_i}.csv")
        rmap = pd.read_csv(f"{HE}/randomise_mapping/{PLURAL[dom]}_batch_{batch_i}_randomise_map.csv")
        randomised = [rmap.iloc[j, 2] for j in range(20)]
        dec, b1, b2 = [], [], []
        for c in range(60):
            v = form.iloc[0, c]
            if   c % 3 == 0: dec.append(1 if v == "Bundle 1 is better" else (2 if v == "Bundle 2 is better" else None))
            elif c % 3 == 1: b1.append(v)
            else:            b2.append(v)
        pairs = []
        for j in range(20):
            yes = (randomised[j] == "yes")                 # yes -> Bundle 1 is LLM4BEAR
            llm = float(b1[j]) if yes else float(b2[j])
            bun = float(b2[j]) if yes else float(b1[j])
            pref = None if dec[j] is None else ("LLM4BEAR" if ((dec[j] == 1) == yes) else "BundleRec")
            pairs.append(dict(human_llm=llm, human_bun=bun, human_pref=pref, batch=batch_i,
                              randomised=randomised[j]))
        out.append(pairs)
    return out

def extract_json(text):
    if text is None: return None
    s, e = str(text).find("{"), str(text).rfind("}")
    if s == -1 or e == -1: return None
    try:    return json.loads(text[s:e+1])
    except Exception:
        try:
            import ast; return ast.literal_eval(text[s:e+1])
        except Exception: return None

def load_claude(dom):
    with open(f"{HE}/{PLURAL[dom]}_claude_zero_shot_responses.pkl","rb") as f:
        responses, mapping = pickle.load(f)
    mods = []
    for k in range(len(responses)):
        d = extract_json(responses[k])
        if not d or "Bundle_A_rating" not in d:
            mods.append(None); continue
        swapped = (mapping[k] == "yes")                    # A is the LLM4BEAR/modified side
        mods.append(d["Bundle_A_rating"] if swapped else d["Bundle_B_rating"])
    return mods

In [6]:
# Build the full joined table of every human-rated surveyed pair
cases = []
for dom in DOMAINS:
    D          = DATA[dom]
    forms      = parse_forms(dom)
    claude_mod = load_claude(dom)
    n_low = 0
    for f_ord, pairs in enumerate(forms):
        for j, pr in enumerate(pairs):
            if pr["human_llm"] in LOW_SCORES: n_low += 1
            P = (pr["batch"] - 1) * 20 + j
            if P >= len(D["cleaned"]):        # pair beyond reconstructed range (shouldn't happen)
                continue
            idx = D["cleaned"][P]
            k   = f_ord * 20 + j
            cases.append(dict(
                dom=dom, batch=pr["batch"], pair=j, randomised=pr["randomised"],
                survey=f"{PLURAL[dom]}_batch_{pr['batch']}",
                human_llm=pr["human_llm"], human_bun=pr["human_bun"], human_pref=pr["human_pref"],
                claude_llm=(claude_mod[k] if k < len(claude_mod) else None),
                llm_rating_orig=D["scores"][0][idx], llm_rating_ref=D["scores"][-1][idx],
                intent=D["intents"][-1][idx],
                orig_ind=D["bundle_indices"][0][idx], final_ind=D["bundle_indices"][-1][idx],
                size=len(D["bundle_indices"][-1][idx]),
            ))
    print(f"{dom:11s}: human rated the LLM4BEAR bundle 1 or 2 in {n_low} pairs")
print("total joined pairs:", len(cases))

clothing   : human rated the LLM4BEAR bundle 1 or 2 in 112 pairs
electronic : human rated the LLM4BEAR bundle 1 or 2 in 115 pairs
food       : human rated the LLM4BEAR bundle 1 or 2 in 87 pairs
total joined pairs: 1740


## 3 · Select the divergence cases

Keep pairs where the **human** scored the LLM4BEAR bundle **1 or 2**, the **LLM4BEAR bundle has 3–4 items**, and
**Claude** also rated it **4 or 5**. Rank by how strongly the two sides disagree and take 5 per domain.

In [7]:
def severity(c):
    cl = c["claude_llm"] if c["claude_llm"] is not None else 0
    return (c["llm_rating_ref"] - c["human_llm"]) + 0.5 * (cl - c["human_llm"])

selected = []
for dom in DOMAINS:
    cands = [c for c in cases
             if c["dom"] == dom
             and c["human_llm"] in LOW_SCORES
             and c["size"] in BUNDLE_SIZES
             and c["claude_llm"] in CLAUDE_HIGH]
    cands.sort(key=severity, reverse=True)
    print(f"{dom:11s}: {len(cands)} candidates -> taking {N_PER_DOMAIN}")
    selected.extend(cands[:N_PER_DOMAIN])
print("selected:", len(selected))

clothing   : 33 candidates -> taking 5
electronic : 34 candidates -> taking 5
food       : 50 candidates -> taking 5
selected: 15


## 4 · Cross-check against the survey each participant actually saw

For every selected case we open the survey JSON, pick the LLM4BEAR side via the randomise marker, and confirm its
displayed image IDs match our reconstructed final bundle — proving the whole join is correct.

In [8]:
def crosscheck(c):
    D = DATA[c["dom"]]
    survey = json.load(open(f"{HE}/surveys/{PLURAL[c['dom']]}_batch_{c['batch']}.json"))
    html   = next(e for e in survey["pages"][c["pair"]]["elements"] if e["type"] == "html")["html"]
    soup   = BeautifulSoup(html, "html.parser")
    is_rand = "no"
    for cm in soup.find_all(string=lambda t: isinstance(t, Comment)):
        if "randomise:" in cm:
            is_rand = cm.split("randomise:")[1].strip().split(";")[0].strip().lower(); break
    cols = soup.find_all("div", style=lambda x: x and "flex:1" in x)
    side = [[drive_id(im.get("src")) for im in col.find_all("img")] for col in cols]
    llm_side  = side[0] if is_rand == "yes" else side[1]       # yes -> Bundle 1 is LLM4BEAR
    recon_ids = [drive_id(idx_to_url(D, i)) for i in c["final_ind"]]
    return set(filter(None, llm_side)) == set(filter(None, recon_ids))

assert all(crosscheck(c) for c in selected), "image cross-check FAILED"
print("image cross-check passed for all", len(selected), "selected cases")

image cross-check passed for all 15 selected cases


## 5 · Summary table

In [9]:
def llm4bear_position(c):
    # randomised == "yes" -> LLM4BEAR was shown as Bundle 1 (left); else Bundle 2 (right)
    return "Bundle 1 (left)" if c["randomised"] == "yes" else "Bundle 2 (right)"

pd.set_option("display.max_colwidth", None)
summary = pd.DataFrame([{
    "domain": c["dom"], "survey": c["survey"], "pair": c["pair"], "size": c["size"],
    "randomised": c["randomised"], "LLM4BEAR_shown_as": llm4bear_position(c),
    "human_LLM4BEAR": c["human_llm"], "human_BundleRec": c["human_bun"], "human_pref": c["human_pref"],
    "claude_LLM4BEAR": c["claude_llm"],
    "LLM4BEAR_self": f"{c['llm_rating_orig']:.2f}->{c['llm_rating_ref']:.2f}",
    "intent": c["intent"],
} for c in selected])
summary

,domain,survey,pair,size,randomised,LLM4BEAR_shown_as,human_LLM4BEAR,human_BundleRec,human_pref,claude_LLM4BEAR,LLM4BEAR_self,intent
0,clothing,clothing_batch_13,15,3,no,Bundle 2 (right),1.0,1.0,BundleRec,5,1.83->4.67,Boys' Casual T-shirt Variety
1,clothing,clothing_batch_13,17,4,no,Bundle 2 (right),1.0,1.0,BundleRec,5,2.50->4.67,Jack Sparrow costume essentials
2,clothing,clothing_batch_34,14,3,no,Bundle 2 (right),1.0,3.0,LLM4BEAR,5,1.67->4.67,Complete Casual Winter Outfit
3,clothing,clothing_batch_21,1,4,no,Bundle 2 (right),1.0,4.0,LLM4BEAR,5,1.67->4.50,Diverse Summer Footwear Options
4,clothing,clothing_batch_13,16,3,yes,Bundle 1 (left),1.0,1.0,LLM4BEAR,4,1.67->4.67,Men's casual t-shirt selection
5,electronic,electronics_batch_23,17,4,no,Bundle 2 (right),1.0,2.0,BundleRec,5,1.83->4.50,Streamlined Audio Connectivity Solutions
6,electronic,electronics_batch_24,0,3,yes,Bundle 1 (left),1.0,1.0,LLM4BEAR,5,3.67->4.50,Premium Earbud Testing Focus
7,electronic,electronics_batch_29,18,3,no,Bundle 2 (right),1.0,1.0,LLM4BEAR,5,1.67->4.50,Diverse Mobile Charging Solutions
8,electronic,electronics_batch_34,17,3,yes,Bundle 1 (left),1.0,2.0,BundleRec,5,4.00->4.50,Comprehensive Apple Charging Solution
9,electronic,electronics_batch_2,8,3,no,Bundle 2 (right),1.0,3.0,BundleRec,4,1.83->4.83,Video and Power Connectivity


## 6 · Case galleries

Each card shows the **BundleRec (original)** bundle next to the **LLM4BEAR (refined)** bundle exactly as the
participant saw them, plus the ratings and the LLM's hidden design intent. An empty markdown cell follows each
gallery — write your analysis of the divergence there.

In [10]:
def _item_block(D, i):
    url  = idx_to_url(D, i)
    ttl  = str(D["titles"][i])
    desc = str(D["descs"][i])
    if len(desc) > 260: desc = desc[:257] + "..."
    return (f"<div style='border-radius:8px;padding:10px;background:#fff;"
            f"box-shadow:0 0 0 1px rgba(0,0,0,.08);margin-bottom:12px;'>"
            f"<img src='{url}' style='width:100%;max-height:220px;object-fit:contain;display:block;margin:0 auto 8px;'>"
            f"<div style='font-weight:700;font-size:14px;line-height:1.25;color:#111;'>{ttl}</div>"
            f"<div style='font-size:12px;line-height:1.35;color:#555;margin-top:4px;'>{desc}</div></div>")

def _chip(label, value, strong=False, bad=False):
    color = "#b00020" if bad else ("#0a7d33" if strong else "#222")
    return (f"<div style='padding:6px 12px;border-radius:8px;background:#fff;"
            f"box-shadow:0 0 0 1px rgba(0,0,0,.08);margin:4px;text-align:center;'>"
            f"<div style='font-size:11px;color:#777;text-transform:uppercase;letter-spacing:.03em;'>{label}</div>"
            f"<div style='font-size:16px;font-weight:700;color:{color};'>{value}</div></div>")

def render_case(c, n=None):
    D = DATA[c["dom"]]
    tag = f"Case {n} &middot; " if n is not None else ""
    header = (f"<div style='font-size:18px;font-weight:800;color:#111;margin-bottom:2px;'>"
              f"{tag}{c['dom'].capitalize()} &middot; Survey <code>{c['survey']}.json</code> &middot; Pair {c['pair']}</div>")
    shown_as = "Bundle 1 (left)" if c["randomised"] == "yes" else "Bundle 2 (right)"
    posnote  = (f"<div style='font-size:12px;color:#777;margin-bottom:6px;'>"
                f"randomised = <b>{c['randomised']}</b> &rarr; in the survey the LLM4BEAR bundle was shown as "
                f"<b>{shown_as}</b>. <i>(Below, BundleRec is always placed on the left for readability.)</i></div>")
    chips = ("<div style='display:flex;flex-wrap:wrap;margin:8px 0 4px;'>"
             + _chip("Human · LLM4BEAR", f"{c['human_llm']:.0f}/5", bad=True)
             + _chip("Human · BundleRec", f"{c['human_bun']:.0f}/5")
             + _chip("Human preferred", c["human_pref"] or "—")
             + _chip("Claude · LLM4BEAR", f"{c['claude_llm']}/5", strong=True)
             + _chip("LLM4BEAR self-rating", f"{c['llm_rating_orig']:.2f} → {c['llm_rating_ref']:.2f}", strong=True)
             + "</div>")
    intent = (f"<div style='background:#fff7e6;border:1px solid #ffd591;border-radius:8px;"
              f"padding:10px 12px;margin:8px 0 14px;font-size:14px;color:#663c00;'>"
              f"<b>LLM4BEAR design intent</b> (never shown to participants): {c['intent']}</div>")
    def column(title, indices, accent):
        items = "".join(_item_block(D, i) for i in indices)
        return (f"<div style='flex:1;min-width:300px;'>"
                f"<h4 style='text-align:center;margin:0 0 8px;color:{accent};'>{title}</h4>{items}</div>")
    body = (f"<div style='display:flex;gap:28px;align-items:flex-start;flex-wrap:wrap;'>"
            + column("BundleRec (original)", c["orig_ind"], "#555")
            + column("LLM4BEAR (refined)",  c["final_ind"], "#0a7d33")
            + "</div>")
    card = (f"<div style='max-width:1000px;margin:0 auto 8px;font-family:system-ui,Arial,sans-serif;"
            f"border:1px solid #e5e5e5;border-radius:12px;padding:16px 18px;'>"
            f"{header}{posnote}{chips}{intent}{body}</div>")
    display(HTML(card))

In [11]:
render_case(selected[0], n=0)

**Case 0 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_


Because it was unreasonable for me to scrape the data off the internet, I had to use Google Search API key as a proxy, which led to some images being poor. I can understand why the participants rated 1/5, as the suit seems out of place, even though the actual item was a Tee.

In [12]:
render_case(selected[1], n=1)

**Case 1 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_

I'm not sure, I definitely agree with the LLM here in this scenario. Potentially it is a participant error.

In [13]:
render_case(selected[2], n=2)

**Case 2 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_

Potentially misleading images, I definitely agree with the LLM.

In [14]:
render_case(selected[3], n=3)

**Case 3 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_

Perhaps this individual thought the larger bundle was better?

In [15]:
render_case(selected[4], n=4)

**Case 4 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_

So I think this is a critical component where the LLM can convince itself that that 3 t-shirts may make up a good bundle, where a human may not agree.

In [16]:
render_case(selected[5], n=5)

**Case 5 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_

This may be a mismatch between human and LLM expertise where there may be a domain gap where it is difficult to evaluate this bundle.

In [ ]:
render_case(selected[6], n=6)

**Case 6 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_

In [17]:
render_case(selected[7], n=7)

**Case 7 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_

Definitely agree with the human participant, the LLM convinced itself that the diversity of chargers were good despite it not making sense.

In [18]:
render_case(selected[8], n=8)

**Case 8 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_

This one I'm on the fence about. I'm not sure who's right.

In [19]:
render_case(selected[9], n=9)

**Case 9 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_

I think this issue is due to the older item/lack of domain expertise of the human participants in the Electronic domain.

In [20]:
render_case(selected[10], n=10)

**Case 10 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_

Without the label it's harder to be convinced of the miscellaneous assortment. So this may be a food domain specific issue.

In [21]:
render_case(selected[11], n=11)

**Case 11 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_

Without the label it's harder to be convinced of the miscellaneous assortment. So this may be a food domain specific issue.

In [22]:
render_case(selected[12], n=12)

**Case 12 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_

Without the label it's harder to be convinced of the miscellaneous assortment. So this may be a food domain specific issue. I do agree with the LLM that the LLM4BEAR bundle is better though.

In [23]:
render_case(selected[13], n=13)

**Case 13 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_

Without the label it's harder to be convinced of the miscellaneous assortment. So this may be a food domain specific issue.

In [24]:
render_case(selected[14], n=14)

**Case 14 — your analysis:**

_Write here: why did the participant rate this LLM4BEAR bundle 1/2 despite the LLM's intent and Claude's high rating?_

Without the label it's harder to be convinced of the miscellaneous assortment. So this may be a food domain specific issue.